In [1]:
import numpy as np
import pandas as pd
import ollama
from datetime import datetime
from tqdm.auto import tqdm

# Load main dataset

In [2]:
# load dataframe
df_kb = pd.read_csv('../data/data-kb.csv', sep='\t', dtype=str) # they are all strings!
df_kb

,pmid,elocationid,title,journal,year,author,affiliation,abstract
0,40938348,doi: 10.1177/2161783X251378518,MazeOut Adaptive Serious Game: Evaluation of P...,Games for health journal,2025,"Alexandre Kira, Rodrigo G Pontes, Augusto K Pe...","Institute of Mathematics and Statistics, Unive...",NaN
1,40938167,doi: 10.1080/00207578.2024.2394076,A reading of stereotypy in autism through the ...,The International journal of psycho-analysis,2025,Leandro Jofré,"Department of Clinical Psychology, Aix Marseil...",Stereotypies currently occupy an important pla...
2,40938164,doi: 10.1080/00207578.2025.2518487,On man who lives once every two times:,The International journal of psycho-analysis,2025,Jan Borowicz,"Polish Psychoanalytical Society, Warszawa, Pol...",The essay delineates a psychic phenomenon of i...
3,40938123,doi: 10.1128/mra.00701-25,Genome sequence of,Microbiology resource announcements,2025,"Zhongsheng Zhu, Jiaojie Guan, Xiuxiu Xie, Chuq...",State Key Laboratory of Quantitative Synthetic...,"Here, we report the draft genome sequence of"
4,40937558,doi: 10.1002/aur.70118,Differences in the Social Experiences of Autis...,Autism research : official journal of the Inte...,2025,"Ellie Roberts, William Mandy, Eirini Flouri","Research Department of Clinical, Educational, ...",Adolescence is a time of complex social and em...
...,...,...,...,...,...,...,...,...
2995,40245385,pii: e63378,Evaluating a Web-Based Application to Facilita...,JMIR research protocols,2025,"Eric Meyer, Hélène Sauzéon, Isabeau Saint-Supe...","GRHAPES (Research group on disability, accessi...",An individual education plan (IEP) is a key el...
2996,40244830,doi: 10.1089/cap.2025.0021,NaN,Journal of child and adolescent psychopharmaco...,2025,"Sally Chu, H Yavuz Ince","Department of Psychiatry, Division of Child an...",NaN
2997,40244560,doi: 10.1007/s12035-025-04929-y,Transcriptomics of Various Diseases Reveals th...,Molecular neurobiology,2025,"Yuxiang Zhang, Junjia Pan, Deqin Zeng, Yifan W...","Key Laboratory of Brain, Cognition and Educati...",Retinal ganglion cells (RGCs) are the only neu...
2998,40244507,doi: 10.1007/s10803-025-06818-8,The Relationship Between Avoidant/Restrictive ...,Journal of autism and developmental disorders,2025,"Borte Gurbuz Ozgur, Buket Canlan Ozaydin, Rabi...","Department of Child and Adolescent Psychiatry,...",The aim is to examine the relationship between...


# Sample PMIDs, initialize ground truth dataframe

In [3]:
# set sample size
sample_size = 100
print(sample_size)

100


In [4]:
# set number of questions per PMID
num_questions_per_pmid = 5
print(num_questions_per_pmid)

5


In [5]:
# sample PMIDs
missing_abstract = df_kb['abstract'].isna()
sampled_pmids = df_kb['pmid'][missing_abstract==False].sample(n=sample_size, \
random_state=824) # sample from those with abstract
sampled_pmids.values

array(['40526487', '40800945', '40426393', '40664085', '40490728',
       '40442917', '40681194', '40594613', '40913732', '40501966',
       '40302609', '40318701', '40529245', '40287634', '40408045',
       '40889034', '40327262', '40373731', '40372284', '40693476',
       '40420291', '40397782', '40564524', '40874214', '40534251',
       '40591708', '40483267', '40771184', '40716154', '40845494',
       '40919359', '40324316', '40452234', '40800483', '40452751',
       '40412004', '40320659', '40804574', '40711707', '40526584',
       '40418536', '40266512', '40247149', '40506004', '40562400',
       '40625349', '40350643', '40896413', '40378475', '40498258',
       '40492449', '40634533', '40723045', '40365400', '40712591',
       '40496650', '40605143', '40502168', '40486672', '40306604',
       '40684358', '40801358', '40457879', '40698409', '40342397',
       '40538592', '40317349', '40330652', '40700716', '40624676',
       '40627091', '40593180', '40796609', '40409375', '405428

In [6]:
# initialize ground truth dataframe
df_synth = pd.DataFrame({'pmid' : sorted(sampled_pmids.to_list()*num_questions_per_pmid), 
                      'ollama_seed' : [i for i in range(num_questions_per_pmid)]*sample_size})
df_synth = df_synth.merge(df_kb, on=['pmid'], how='left')[['pmid', 'ollama_seed', 'abstract']]
df_synth # has seed for reproducibility

,pmid,ollama_seed,abstract
0,40247149,0,Functional magnetic resonance imaging (fMRI) h...
1,40247149,1,Functional magnetic resonance imaging (fMRI) h...
2,40247149,2,Functional magnetic resonance imaging (fMRI) h...
3,40247149,3,Functional magnetic resonance imaging (fMRI) h...
4,40247149,4,Functional magnetic resonance imaging (fMRI) h...
...,...,...,...
495,40933682,0,As autistic students enter postsecondary educa...
496,40933682,1,As autistic students enter postsecondary educa...
497,40933682,2,As autistic students enter postsecondary educa...
498,40933682,3,As autistic students enter postsecondary educa...


# Use LLM to generate synthetic questions

In [7]:
# set LLM handle
model_handle = 'llama3.2:1b'
print(model_handle)

llama3.2:1b


In [8]:
# make prompt template for synthetic quesitons
prompt_template = """
You are a layperson who is interested in autism spectrum disorders.
Write a general question about autism that can be answered by the ABSTRACT below.
Keep the question only one or two sentences long.
Just write the question itself; do not write anything else.

PASSAGE:
{abstract}
""".strip()
print(prompt_template)

You are a layperson who is interested in autism spectrum disorders.
Write a general question about autism that can be answered by the ABSTRACT below.
Keep the question only one or two sentences long.
Just write the question itself; do not write anything else.

PASSAGE:
{abstract}


In [9]:
def generate_question(abstract, seed):
    prompt_text = prompt_template.format(abstract=abstract)
    response = ollama.chat(model=model_handle, messages=[{'role' : 'user', 'content' : prompt_text}],
                          options={'seed' : seed})
    return response['message']['content'].strip()

In [10]:
# demo question generation
demo_abstract = df_synth.iloc[0]['abstract']
print(demo_abstract)
print()
print(generate_question(demo_abstract, seed=42))

Functional magnetic resonance imaging (fMRI) has emerged as a transformative tool in analyzing and understanding brain diseases. It is a challenge to learn effective features from the high-dimensional fMRI. Most studies have focused on extracting connectivity-based features for disease analysis. However, they heavily rely on the software toolboxes to construct connectivity-based features, which may suffer from large errors because of different manual parameter settings and thus lead to bad performance in brain disorder analysis.

What is one potential limitation of using fMRI to analyze autism?


In [11]:
def generate_questions_list(df):
    records = df.to_dict(orient='records')
    synthetic_questions = [generate_question(record['abstract'], record['ollama_seed']) \
    for record in tqdm(records)]
    return synthetic_questions

In [12]:
# generate synthetic
print(datetime.now())
df_synth['synthetic_question'] = generate_questions_list(df_synth)
print(datetime.now())

2025-09-12 22:14:58.721408


  0%|          | 0/500 [00:00<?, ?it/s]

2025-09-12 22:30:31.158755


In [13]:
# to demonstrate reproducibility, generate again
if False: # set to True to generate again and compare
    print(datetime.now())
    demo_reproducibility = pd.Series(generate_questions_list(df_synth))
    print(datetime.now())
    print((df_synth['synthetic_question']==demo_reproducibility).value_counts()) # all true if reproducible
    print(pd.concat([df_synth['synthetic_question'], demo_reproducibility], axis=1))

In [14]:
# finalize ground truth dataframe
df_synth = df_synth[['pmid', 'ollama_seed', 'synthetic_question']]
df_synth

,pmid,ollama_seed,synthetic_question
0,40247149,0,What role do computer algorithms play in devel...
1,40247149,1,Is it more efficient to use machine learning a...
2,40247149,2,What role do researchers believe software tool...
3,40247149,3,What role do expert systems play in improving ...
4,40247149,4,What is one potential limitation of using fMRI...
...,...,...,...
495,40933682,0,What is a major challenge that autistic studen...
496,40933682,1,Does a lack of visibility and understanding fr...
497,40933682,2,What drives college students with autism to de...
498,40933682,3,What is the primary reason that autistic colle...


In [15]:
# look at a few questions
print('\n'.join(df_synth['synthetic_question'].to_list()[:10]))

What role do computer algorithms play in developing effective fMRI features for diagnosing autism spectrum disorders?
Is it more efficient to use machine learning algorithms to learn the underlying patterns in fMRI data rather than relying solely on traditional statistical analysis methods?
What role do researchers believe software tools play in shaping the accuracy of fMRI-based brain disease analysis?
What role do expert systems play in improving the accuracy of fMRI data analysis?
What is one potential limitation of using fMRI data in autism research that has been observed or hypothesized by researchers?
What are the differences between the brain function of individuals with autism spectrum disorders (ASD) compared to those without it?
Here's a general question about autism that can be answered by the abstract:

What is the primary characteristic or feature that distinguishes individuals with autism from those without it, based on research findings?
What are the key differences betw

# Write CSV

In [16]:
# write CSV file
df_synth.to_csv('../data/data-synth-question.csv', index=False, sep='\t')

In [17]:
print(datetime.now())

2025-09-12 22:30:31.195624
